# Đo tốc độ tầng kỹ trên Kaggle (phase 2g)

Notebook này đo bao nhiêu mili giây mỗi ảnh cắt khi chạy YOLO11x trên GPU T4, rồi chạy thử cả đường ống trên một video.

**Trước khi chạy:**
1. Bên phải, mở **Settings → Accelerator** và chọn **GPU T4 x2**.
2. Bật **Internet** (Settings → Internet on). Kaggle sẽ đòi xác minh số điện thoại một lần.
3. Thêm video: **Add Input → Upload** thành một Dataset, giữ chế độ **Private**. Sửa biến `VIDEO` ở ô 2 cho đúng đường dẫn.
4. Nếu repo GitHub là private: **Add-ons → Secrets**, thêm secret tên `GITHUB_TOKEN` (token chỉ cần quyền đọc repo). Repo công khai thì bỏ qua bước này.

Chạy lần lượt từng ô. Ô cuối ghi `/kaggle/working/bench.json` và các file timing; gửi lại các file đó cho Nguyên.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

BRANCH = "claude/project-thread-g4frkm"
REPO = "github.com/vuthanhhnguyenn/httm"
VIDEO = "/kaggle/input/<dataset>/EPf9O7UXRoM.mp4"  # sửa cho đúng tên Dataset của bạn
WORK = Path("/kaggle/working")
REPO_DIR = WORK / "httm"

token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # không có secret: clone bằng URL công khai

def sh(cmd, cwd=None):
    print("$", cmd.replace(token, "***") if token else cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

assert Path(VIDEO).is_file(), f"Không thấy video: {VIDEO} (sửa biến VIDEO)"
sh("nvidia-smi -L")

In [ ]:
# Clone nhánh và cài những gói còn thiếu (Kaggle đã có torch, opencv, numpy). Repo không đóng gói pip nên
# các script tự thêm packages/*/src vào sys.path; không cần pip install -e.
url = f"https://{token}@{REPO}.git" if token else f"https://{REPO}.git"
if not REPO_DIR.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} {url} {REPO_DIR}")
sh('pip install -q "ultralytics==8.4.160" pyyaml "pydantic>=2.9,<3.0"')

In [ ]:
# Tải model YOLO11x (kiểm SHA-256 theo models/manifest.yaml)
sh("python scripts/download_model.py", cwd=REPO_DIR)

In [ ]:
# Đo tốc độ: batch 8/16/32, FP16 tắt/bật, card 0 và 1, imgsz 960/640
sh(f"python scripts/bench_review.py {VIDEO} --out {WORK}/bench.json --batches 8,16,32 --halves off,on "
   f"--devices 0,1 --image-sizes 960,640", cwd=REPO_DIR)

In [ ]:
# Chạy thử đường ống trên video: tầng nhanh rồi tầng kỹ, đo thời gian từng bước
timings = {}
def timed(name, cmd):
    start = time.time()
    sh(cmd, cwd=REPO_DIR)
    timings[name] = round(time.time() - start, 1)

fast_out, review_out = WORK / "p2g-fast", WORK / "p2g-review"
timed("fast_tier_seconds", f"python scripts/analyze_video_multi.py {VIDEO} --out {fast_out} --scene cctv --mode full "
                           f"--fps 5 --suspects --no-video --overwrite")
timed("review_tier_seconds", f"python scripts/review_suspects.py {VIDEO} {fast_out}/suspects.jsonl --out {review_out} "
                             f"--fps 5 --max-frames 0 --device 0")
(WORK / "timings.json").write_text(json.dumps(timings, indent=2), encoding="utf-8")

In [ ]:
# Tóm tắt: bảng đo tốc độ, hai thời gian, và các file cần gửi lại
sys.path.insert(0, str(REPO_DIR / "scripts"))
from bench_review import format_table

print(format_table(json.loads((WORK / "bench.json").read_text(encoding="utf-8"))))
print()
print(f"Tầng nhanh : {timings['fast_tier_seconds']} s")
print(f"Tầng kỹ    : {timings['review_tier_seconds']} s")
for src, dst in ((fast_out / "timing.json", WORK / "timing-fast.json"), (review_out / "timing.json", WORK / "timing-review.json")):
    if src.is_file():
        dst.write_text(src.read_text(encoding="utf-8"), encoding="utf-8")
print()
print("Gửi lại cho Nguyên:", *sorted(str(p) for p in WORK.glob("*.json")), sep="\n  ")